# This notebook extracts CMG simulations results and save them as Numpy arrays
* File format conversion: CMG sr3/gmch.sr3 --> CMG rwo (by running CMG Results Report software on rwd files) --> numpy array. sr3 is the binary CMG simulation result file, whereas rwo is the extracted info in the ascii format. rwd is a configuration file to tell CMG Results Report software how to export results
* For Vienna goethermal, geomechanical simulations are stored in sr3 files, not in separate gmch.sr3 files.
* **This version uses `CMG2npy_robust`** instead of `CMG2npy`. The rwo file is read with a single `read()` rather than iterating ~700k lines, which avoids the silent partial reads seen when parsing off the Oak drive mapped over SMB (`Warning: Expected 139 values, got 76 ...`). Those rows used to be skipped and left as zeros; now any incomplete read raises instead.
* Set `expected_shape` and `expected_active` below so a short read is caught rather than passing silently.

# Step 0: Run this cell to provide inputs for all following steps

In [ ]:
import numpy as np
from pathlib import Path

current_path = Path('.')

################## User Inputs ############################## 
name_prefix = '260812' # file name prefix 
n_cases = 80 # number of simulation cases
property_list = ['STRESMXP','STRESMNP','STRESINT','PRES','TEMP'] # list of properties (CMG keywords) to extract from the simulation results
sim_results_folder_path = current_path/f'{name_prefix}_sherlock' # path to the simulation results folder
sim_results_file_format = 'sr3' # sr3 or gmch.sr3
save_folder_path = sim_results_folder_path/f'{name_prefix}_sim_py' # path to save extracted results

# integrity checks (used by CMG2npy_robust to catch an incomplete read)
expected_shape = (139, 248, 23) # (n_i, n_j, n_k) of the JD_geothermal grid
expected_active = 401735 # number of non-zero cells at the first time step, taken from a known-good case
copy_local_first = False # set True to copy each rwo to local disk before parsing, if the share is unreliable
wait_for_output = True # wait for each rwo to finish being written before moving on

# storage type of the saved npy files, used by every option below
# CMG writes 4 significant digits (*PRECISION 4) and float32 holds about 7,
# so nothing real is lost and the files are half the size (38 MB -> 19 MB each).
# Set to None to keep the float64 default.
# Do not mix the two in one output folder: an old float64 array and a new
# float32 array of the same case will not compare equal with np.array_equal.
npy_dtype = np.float32
################## End of User Inputs #######################

# Step 1: Convert CMG sr3/gmch.sr3 files to rwo files

Note this part needs to be run on a machine where CMG Results Report software is installed and the CMG2npy_robust source code file.

Two options below:
* **Option A (in place)** — the original workflow, one rwd per case per property, everything written on the simulation folder. Use this if that folder is already local.
* **Option B (local disk, recommended when the sr3 files sit on a mapped network drive)** — stages each case on local disk, runs Report.exe there, parses locally, and writes back only the npy. This keeps the sr3 reads and the large rwo files off the share. Option B does Step 1 and Step 2 together, so skip Step 2 if you use it.

Requesting several properties from one rwd was tried and does not work with CMG Results 2024.20 as invoked here, so one rwd per property is used in both options.

In [ ]:
from pathlib import Path
from CMG2npy_robust import generate_CMG_rwd, run_CMG_rwd_report
from tqdm import tqdm

current_path = Path('.')

################## User Inputs ############################## 
# set in Step 0
################## End of User Inputs #######################

failed = []
for property in property_list:
    for case_num in tqdm(range(1,n_cases+1),desc=f'Converting sr3 to rwo for keyword {property}'):
        try:
            generate_CMG_rwd(
                sr3_folder_path = sim_results_folder_path,
                case_name = f'case{case_num}',
                property = property,
                sim_results_file_format = sim_results_file_format,
                precision = 4
            )

            # the Report.exe exit code is checked, and passing `property` makes
            # this wait until the rwo has finished being written
            run_CMG_rwd_report(
                rwd_folder_path = sim_results_folder_path,
                case_name = f'case{case_num}',
                property = property,
                cmg_version = 'ese-ts2win-v2024.20',
                wait_for_output = wait_for_output,
            )
        except Exception as e:
            failed.append((case_num, property, str(e)))
        finally:
            # remove rwd file
            rwd = Path(sim_results_folder_path, f'case{case_num}.rwd')
            if rwd.exists():
                rwd.unlink()

print("\nFinished generating rwo files for all cases.")
if failed:
    print(f"\n{len(failed)} case/property combinations FAILED:")
    for c,p,e in failed:
        print(f"  case{c} {p}: {e}")

### Option B: stage each case on local disk (Step 1 + Step 2 combined)

Running in place on a mapped drive moves roughly 1.1 GB across the network per case for 5 properties: the sr3 is read once per property, and every rwo is written to the share and then read straight back to be parsed. Staging locally leaves only the sr3 copy in and the npy files out.

Properties are handled one at a time and each rwo is deleted once parsed, so local disk use stays around sr3 + one rwo. Set `local_work_dir` to a fast local path; `None` uses the system temp directory.

In [ ]:
import numpy as np
from pathlib import Path
from CMG2npy_robust import extract_case_local
from tqdm import tqdm

################## User Inputs ############################## 
local_work_dir = None # None uses the system temp dir; otherwise a fast LOCAL path, e.g. r'C:\cmg_work'
# npy_dtype set in Step 0
################## End of User Inputs #######################

save_folder_path.mkdir(parents=True, exist_ok=True)

failed = []
for case_num in tqdm(range(1,n_cases+1), desc='sr3 -> rwo -> npy on local disk'):
    try:
        extract_case_local(
            sr3_folder_path = sim_results_folder_path,
            case_name = f'case{case_num}',
            property_list = property_list,
            save_folder_path = save_folder_path,
            sim_results_file_format = sim_results_file_format,
            precision = 4,
            cmg_version = 'ese-ts2win-v2024.20',
            local_work_dir = local_work_dir,
            expected_shape = expected_shape,
            expected_active = expected_active,
            dtype = npy_dtype,
            show_info = False,
        )
    except Exception as e:
        failed.append((case_num, str(e)))

print("\nFinished extracting all cases to numpy arrays.")
if failed:
    print(f"\n{len(failed)} cases FAILED:")
    for c,e in failed:
        print(f"  case{c}: {e}")
    print("\nCases that terminated early have fewer time steps; those are expected to differ, not fail.")

# Step 2: Extract simulation results from rwo files into numpy arrays

## Option A: Extract results on all grid cells

In [ ]:
from pathlib import Path
from CMG2npy_robust import CMG_rwo2npy
from tqdm import tqdm

current_path = Path('.')

################## User Inputs ############################## 
# set in Step 0
################## End of User Inputs #######################

save_folder_path.mkdir(parents=True, exist_ok=True)

failed = []
for property in property_list:
    for case_num in tqdm(range(1,n_cases+1), desc=f'Generating numpy arrays for keyword {property}'):
        try:
            sim_results = CMG_rwo2npy(
                rwo_folder_path = sim_results_folder_path/'rwo',
                case_name = f'case{case_num}',
                property = property,
                is_save = True,
                save_folder_path = save_folder_path,
                show_info = False,
                expected_shape = expected_shape,
                expected_active = expected_active,
                copy_local_first = copy_local_first,
                dtype = npy_dtype,
            )
        except Exception as e:
            # an incomplete read now raises instead of silently leaving zeros;
            # collect the failures so the whole batch does not stop on one case
            failed.append((case_num, property, str(e)))

print("\nFinished generating numpy arrays for all cases.")
if failed:
    print(f"\n{len(failed)} case/property combinations FAILED and were not saved:")
    for c,p,e in failed:
        print(f"  case{c} {p}: {e}")
    print("\nRe-run these; a repeated failure means the rwo itself is short (e.g. an early-terminated run).")

## Step 3: Check the extracted arrays before using them

Cases that terminated early (for example the thermal-front convergence failures) have fewer time steps than the rest. Downstream notebooks that assume a fixed `n_times` will fail on those, so list the shapes here first.

In [ ]:
import numpy as np
from pathlib import Path

################## User Inputs ############################## 
# set in Step 0
################## End of User Inputs #######################

shapes = {}
missing = []
for case_num in range(1,n_cases+1):
    f = save_folder_path/f'case{case_num}_{property_list[0]}.npy'
    if not f.exists():
        missing.append(case_num); continue
    a = np.load(f, mmap_mode='r')
    shapes.setdefault(a.shape, []).append(case_num)

print(f'Shapes found in {save_folder_path}:')
for s, cases in sorted(shapes.items(), key=lambda kv: -len(kv[1])):
    print(f'  {s}: {len(cases)} cases  e.g. {cases[:8]}')
if missing:
    print(f'\nMISSING (not extracted): {missing}')

if len(shapes) > 1:
    n_times_ref = max(shapes, key=lambda s: len(shapes[s]))[-1]
    odd = [c for s, cases in shapes.items() if s[-1] != n_times_ref for c in cases]
    print(f'\nWARNING: cases with a different number of time steps: {sorted(odd)}')
    print('Exclude these downstream, or derive n_times per case instead of fixing it.')
else:
    print('\nAll extracted cases share the same shape.')